# LeKiwi Gripper Setup — flash EPROM & hardcode calibration

One-time setup for the LeKiwi gripper servo (Feetech STS3215, motor id 6, here named `pincopen`).
It flashes safety/tuning parameters and the travel limits into the motor's **EPROM**, then hardcodes
the `0..100` gripper calibration that `lerobot` uses for normalized open/close.

Install the hardware deps with `pip install -e ".[hardware]"`, then connect the gripper servo over USB.

**How to use:**
1. Edit the **connection cell** (`port`, `motor_id`) and the **angle limits** (`MIN_ANGLE_DEG` / `MAX_ANGLE_DEG`) for your build.
2. Run the cells **top to bottom** — do *not* "Run All". Two steps put you at the hardware: the MIN/MAX limit
   flash has a mandatory hand-check (release torque, move the jaws fully **OPEN** → expect ≈ −135°, fully
   **CLOSED** → expect ≈ 0°) **before** you flash.
3. EPROM writes persist across power cycles — run this once per motor (or whenever you change the values).

> ⚠️ These cells write to the motor's EPROM. Confirm the OPEN/CLOSED angle readings match your gripper's
> mounting before flashing the position limits; adjust the angle convention if they don't.

### SMS/STS Servo - eManual
http://doc.feetech.cn/#/prodinfodownload?srcType=FT-SMS-STS-emanual-229f4476422d4059abfb1cb0


In [ ]:
from lerobot.motors.feetech import FeetechMotorsBus
from lerobot.motors import Motor, MotorNormMode

motor_name = "pincopen"
motor_id = 6  # Set the ID of the motor, it should be configured in its firmware before the assembly.

bus = FeetechMotorsBus(
    port='/dev/ttyACM0',
    motors={
        motor_name: Motor(motor_id, "sts3215", MotorNormMode.RANGE_0_100)  # 0-100 is intuitive for a gripper!
    },
)

print(bus)
bus.connect()
print("Connected!")

### Reading current EPROM (flash) settings

In [ ]:
parameters = [
    # Identity & comms
    "ID",
    "Baud_Rate",
    # Firmware
    "Firmware_Major_Version",
    "Firmware_Minor_Version",
    # Calibration
    "Homing_Offset",
    "Min_Position_Limit",
    "Max_Position_Limit",
    # Mode & lock
    "Operating_Mode",
    "Lock",
    # Motion / PID
    "Acceleration",
    "P_Coefficient",
    "I_Coefficient",
    "D_Coefficient",
    # Torque & protection
    "Max_Torque_Limit",
    "Overload_Torque",
    "Protective_Torque",
    "Protection_Time",
    # Voltage limits
    "Max_Voltage_Limit",
    "Min_Voltage_Limit",
    # Live telemetry (RAM, not EPROM)
    "Present_Voltage",
    "Present_Position",
]

for param in parameters:
    print(f"{param}:", bus.read(param, motor_name, normalize=False))

### Motor Configuration (EPROM settings) — Run once to "flash" the motor

These are the safety parameters you want written to the motor's EPROM. 

This is done once (or whenever you need to change them), not during every calibration. 

You can do this with a simple script using the FeetechMotorsBus:


In [ ]:
def configure_eprom(bus, motor, settings: dict, *, verify=True, disable_torque=False):
    """Write EPROM/config registers safely: (optionally drop torque) -> unlock -> write -> always re-lock.

    settings: {register_name: raw_value}, e.g. {"Acceleration": 200, "Max_Torque_Limit": 1000}
    Use raw values only (normalize=False). Do NOT use this for Goal_Position.

    Notes:
      - Don't put "ID" here: changing the id breaks name-based access until you reconnect.
      - disable_torque=True is safer for critical writes, but leaves the motor limp afterward
        (re-enable with bus.enable_torque(motor) before commanding positions).
    """
    if disable_torque:
        bus.write("Torque_Enable", motor, 0, normalize=False)

    bus.write("Lock", motor, 0, normalize=False)          # unlock EPROM
    try:
        for reg, val in settings.items():
            bus.write(reg, motor, val, normalize=False)
            print(f"  set {reg} = {val}")
    finally:
        bus.write("Lock", motor, 1, normalize=False)      # re-lock even if a write fails

    if verify:
        print("verify:")
        for reg, val in settings.items():
            got = bus.read(reg, motor, normalize=False)
            print(f"  {reg}: {got} (expected {val}) {'OK' if got == val else 'MISMATCH'}")

`Max_Torque_Limit = 1000` (full 100%) but `Overload_Torque = 40` (trips at 40%). These govern different regimes:
- Max_Torque is the instantaneous ceiling, so the jaws get full torque to close fast through empty air.
- Overload governs sustained load. The moment the jaws actually press on something at >40% for the protection window, it trips.

So the design is "move freely and quickly, but the instant you're loaded, back off." That's exactly what you want from a gripper.

**Protection_Time = 7 (70 ms) and Protective_Torque = 5 (5%)** complete the soft-stop: contact is detected within 70 ms, then torque collapses to a 5% feather-hold. Probable reasons to want it that aggressive:
- **Don't crush**. Fragile or light objects (and fingers) survive. The gripper "gives up" almost on contact.
- **Thermal/current self-protection**. A gripper is the joint most likely to sit stalled while holding something. At full torque against a stall the STS3215 pulls ~2 to 2.5 A and heats toward the 70°C cutoff. Dropping to 5% within 70 ms
avoids sustained stall current during long holds.
- **Gear protection**. Backing off fast spares the plastic/metal gear train from shock-loading at full torque.

`Acceleration = 200` (vs default 0): a velocity ramp so the jaws ease into motion instead of snapping. With hard travel limits set (512–2048), a ramp keeps it from slamming into the end stops or whacking an object on approach.
Smoother, less overshoot, less mechanical shock.

In [ ]:
# Tuning / safety EPROM params. Running this WRITES them (they were no-ops/commented before).
# Format: value  # range | unit | factory default | what it does
# (defaults below are this motor's readout, matching the Feetech SMS/STS datasheet)
configure_eprom(bus, motor_name, {
    "Acceleration":      200,   # 0-254  | x100 steps/s^2   | default 0          | goal-move accel/decel ramp (0 = instant)
    "Max_Torque_Limit":  1000,  # 0-1000 | 0.1% (1000=100%) | default 1000       | hard cap on output torque (1000 = no change)
    "Overload_Torque":   40,    # 0-100  | % of max torque  | default 80         | load above this for Protection_Time -> overload trip
    "Protective_Torque": 5,     # 0-100  | % of max torque  | default 20         | torque held after an overload trip
    "Protection_Time":   7,     # 0-254  | x10 ms           | default 200 (=2s)  | overload must persist this long before tripping (7 = 70 ms)
})

### Flash MIN/MAX Position Limit

1. Define the angle convention + helpers, and print the target raw limits.
2. **Sanity-check by hand first:** release torque, move the gripper fully OPEN and confirm it reads about `-135°`, then fully CLOSED and confirm about `0°`.
3. Only if both readings look right, flash the limits into EPROM.

In [ ]:
# Your known limits (from pypot)
MIN_ANGLE_DEG = -135  # fully open
MAX_ANGLE_DEG = 0     # fully closed

# STS3215 / STS3250: 4096 steps = 360 degrees, center at 2048
def deg_to_raw(deg: float) -> int:
    return int(round(2048 + deg * 4096 / 360))

def raw_to_deg(raw: int) -> float:
    return (raw - 2048) * 360 / 4096

def read_deg(motor=motor_name):
    """Read Present_Position and show it in degrees (same center-2048 convention as deg_to_raw)."""
    raw = bus.read("Present_Position", motor, normalize=False)
    deg = raw_to_deg(raw)
    print(f"Present_Position: {raw} raw  ->  {deg:6.1f} deg")
    return deg

range_min = deg_to_raw(MIN_ANGLE_DEG)  # ~512
range_max = deg_to_raw(MAX_ANGLE_DEG)  # ~2048
print(f"Target limits to flash: {range_min} ({MIN_ANGLE_DEG} deg)  ->  {range_max} ({MAX_ANGLE_DEG} deg)")

In [ ]:
# Sanity-check the angle convention BEFORE flashing.
# Release torque so you can move the jaws by hand, then run this once per position:
bus.disable_torque(motor_name)

read_deg()
# 1) move gripper FULLY OPEN by hand, run this   -> expect about -135 deg
# 2) move gripper FULLY CLOSED by hand, re-run    -> expect about    0 deg
# If a reading is far off, the mounting/zero (Homing_Offset) differs from this
# convention -- fix that (or set Homing_Offset = 0) before flashing limits.

In [ ]:
# Only once OPEN reads ~ -135 deg and CLOSED reads ~ 0 deg:
# flash the hardware travel limits into EPROM (configure_eprom unlocks, writes, re-locks).
configure_eprom(bus, motor_name, {
    "Min_Position_Limit": range_min,
    "Max_Position_Limit": range_max,
})

### Hardcode the gripper calibration

In [ ]:
from lerobot.motors import MotorCalibration

# Register the in-memory calibration used for 0-100 normalization (move_to / read_pos).
# write_calibration writes by motor name; the id field is metadata, it does NOT renumber the motor.
calibration = {
    motor_name: MotorCalibration(
        id=motor_id,
        drive_mode=0,           # 0=normal, 1=inverted
        homing_offset=0,        # raw offset from physical zero
        range_min=range_min,    # -135 deg (fully open)
        range_max=range_max,    # 0 deg (fully closed)
    )
}

print("Writing calibration to motor...")
bus.write_calibration(calibration)
print("Calibration written successfully.")

### Test gripper movement

`move_to(target)` commands a normalized position and reads back where it landed.

`open_gripper()` / `close_gripper()` drive to the range ends. Units are `0..100` as gripper is `RANGE_0_100`

In [ ]:
import time


def read_pos(motor=motor_name):
    """Current position in normalized units."""
    return bus.read("Present_Position", motor, normalize=True)


def _span(motor=motor_name):
    """(0%, 100%) ends in normalized units, from the motor's norm mode."""
    return (0, 100) if bus.motors[motor].norm_mode is MotorNormMode.RANGE_0_100 else (-100, 100)


def move_to(target, motor=motor_name, wait=0.5, report=True):
    """Command a normalized target, wait, then read back where it landed."""
    bus.write("Goal_Position", motor, float(target), normalize=True)
    time.sleep(wait)
    actual = read_pos(motor)
    if report:
        print(f"target {target:6.1f} -> actual {actual:6.1f}")
    return actual


def open_gripper(motor=motor_name):
    return move_to(_span(motor)[0], motor)


def close_gripper(motor=motor_name):
    return move_to(_span(motor)[1], motor)


print(f"current: {read_pos():.1f}")

In [ ]:
move_to(60, motor_name)

In [ ]:
open_gripper()

In [ ]:
close_gripper()